In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torch.optim as optim


In [2]:

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


In [3]:

# Dataset class to load images and CSV features
class ImageFeatureDataset(Dataset):
    def __init__(self, images_dir, csv_file, transform=None):
        self.images_dir = images_dir
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Get image_id and features
        image_id = self.data.iloc[idx]['image_id']
        image_path = os.path.join(self.images_dir, f"{image_id}.jpg")

        # Load and transform image
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Load CSV features as tensor
        csv_features = self.data.iloc[idx, 1:-1].values.astype(np.float32)
        label = self.data.iloc[idx]['label'] if 'label' in self.data.columns else None

        return {
            "image": image,
            "csv_features": torch.tensor(csv_features),
            "label": torch.tensor(label) if label is not None else None
        }


In [4]:

# Multimodal Model: ResNet for image features + Fully Connected for CSV features
class MultimodalModel(nn.Module):
    def __init__(self, image_feature_dim, csv_feature_dim, combined_feature_dim, num_classes):
        super(MultimodalModel, self).__init__()
        
        # ResNet for Image Features
        self.resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])  # Remove last FC layer

        # FC layers for CSV Features
        self.csv_fc = nn.Sequential(
            nn.Linear(csv_feature_dim, 128),
            nn.ReLU(),
            nn.Linear(128, combined_feature_dim)
        )

        # Fusion layer for combined features
        self.fc1 = nn.Linear(image_feature_dim + combined_feature_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, image, csv_features):
        # Extract image features from ResNet
        image_features = self.resnet(image).view(image.size(0), -1)  # Flatten

        # Process CSV features through FC layers
        csv_features = self.csv_fc(csv_features)

        # Concatenate image and CSV features
        combined_features = torch.cat((image_features, csv_features), dim=1)

        # Final FC layers
        x = torch.relu(self.fc1(combined_features))
        x = self.fc2(x)
        
        return x, combined_features  # Returning combined features for extraction


In [ ]:

# Parameters and setup
images_dir = r"C:\Users\pavan\Downloads\images"
csv_file = r"C:\Users\pavan\Downloads\articles.csv\articles_clean.csv"
num_classes = 10                       # Adjust as needed
image_feature_dim = 2048                # ResNet-50 output feature size
csv_feature_dim = len(dataset[0]["csv_features"])
combined_feature_dim = 128
batch_size = 32
num_epochs = 10


In [ ]:

# Initialize Dataset and DataLoader
dataset = ImageFeatureDataset(images_dir=images_dir, csv_file=csv_file, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [ ]:

# Initialize Model, Optimizer, and Loss Function
model = MultimodalModel(image_feature_dim=image_feature_dim, csv_feature_dim=csv_feature_dim, combined_feature_dim=combined_feature_dim, num_classes=num_classes).cuda()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


In [ ]:

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch in dataloader:
        images = batch["image"].cuda()
        csv_features = batch["csv_features"].cuda()
        labels = batch["label"].cuda()

        optimizer.zero_grad()
        outputs, _ = model(images, csv_features)  # `_` is the combined features
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(dataloader):.4f}")

print("Training complete.")


In [ ]:

# Feature Extraction
def extract_features(dataloader, model):
    model.eval()
    features = []
    with torch.no_grad():
        for batch in dataloader:
            images = batch["image"].cuda()
            csv_features = batch["csv_features"].cuda()
            _, combined_features = model(images, csv_features)
            features.append(combined_features.cpu().numpy())

    return np.concatenate(features, axis=0)

# Get extracted features for the entire dataset
final_features = extract_features(dataloader, model)
print("Extracted feature shape:", final_features.shape)
